In [0]:
# Databricks Notebook
# MAGIC %md # Gold Layer - Star Schema Transformation

# COMMAND ----------
# IMPORT CÁC THƯ VIỆN CẦN THIẾT
from datetime import timedelta
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Khởi tạo schema gold nếu chưa tồn tại
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql("USE CATALOG data_dev_olist")
# COMMAND ----------
# MAGIC %md ## 1. Dimension Tables

# COMMAND ----------
# DBTITLE 1, dim_customer
# Đọc dữ liệu từ tầng Silver (Giả định bạn đã lưu silver dạng bảng trong metastore)
silver_cleaned_customer = spark.table("silver.clean_customer")
silver_cleaned_geolocation = spark.table("silver.clean_geolocation")

# Biến đổi dữ liệu
customer = silver_cleaned_customer
geolocation = silver_cleaned_geolocation

joined_customer = customer.join(
    geolocation,
    customer["customer_zip_code_prefix"] == geolocation["geolocation_zip_code_prefix"],
    how="left"
)

joined_customer = (joined_customer
    .withColumnRenamed("geolocation_lat", "customer_lat")
    .withColumnRenamed("geolocation_lng", "customer_lng")
    .withColumn("customer_city", col("geolocation_city"))
    .withColumn("customer_state", col("geolocation_state"))
    .drop("geolocation_city", "geolocation_state", "customer_zip_code_prefix", "geolocation_zip_code_prefix")
    .dropDuplicates(subset=["customer_id"])
)

dim_customer_df = joined_customer.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_lat",
    "customer_lng"
)

# Ghi thẳng vào Delta Lake của Databricks
dim_customer_df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_customer")

# COMMAND ----------
# DBTITLE 1, dim_seller
silver_cleaned_seller = spark.table("silver.clean_seller")

dim_seller_df = silver_cleaned_seller.select(
    "seller_id",
    "seller_zip_code_prefix"
)

dim_seller_df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_seller")

# COMMAND ----------
# DBTITLE 1, dim_review
silver_cleaned_order_review = spark.table("silver.clean_order_review")

dim_review_df = silver_cleaned_order_review.select(
    "review_id",
    "review_score"
)

dim_review_df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_review")

# COMMAND ----------
# DBTITLE 1, dim_product
silver_cleaned_product = spark.table("silver.clean_product")
silver_cleaned_product_category = spark.table("silver.clean_product_category")

dim_product_df = silver_cleaned_product.join(
    silver_cleaned_product_category,
    "product_category_name",
    "inner"
).select(
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

dim_product_df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_product")

# COMMAND ----------
# DBTITLE 1, dim_order
silver_cleaned_order = spark.table("silver.clean_order")
silver_cleaned_payment = spark.table("silver.clean_payment")

dim_order_df = silver_cleaned_order.join(
    silver_cleaned_payment, "order_id", "inner"
).select(
    "order_id",
    "order_status",
    "payment_type"
)

dim_order_df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_order")

# COMMAND ----------
# COMMAND ----------
# DBTITLE 1, dim_date    
# Trích xuất trực tiếp từ bảng clean_order thay vì bảng 'silver.date' không tồn tại
silver_clean_order = spark.table("silver.clean_order")

# Lấy khoảng ngày biên (Min/Max) từ timestamp đơn hàng
start_date_val = silver_clean_order.select(min("order_purchase_timestamp")).first()[0]
end_date_val = silver_clean_order.select(max("order_purchase_timestamp")).first()[0]

# Chuyển đổi timestamp thành date (loại bỏ giờ phút giây để tối ưu cho Dim Date)
start_date = start_date_val.date() if start_date_val else None
end_date = end_date_val.date() if end_date_val else None

if start_date and end_date:
    # Tính số lượng ngày cần tạo trong sequence
    days_count = (end_date - start_date).days + 1
    
    # Tạo sequence ngày sử dụng hàm native Spark range
    date_df = spark.range(0, days_count).withColumn(
        "full_date", 
        lit(start_date) + col("id").cast("string").cast("interval day")
    )

    # Thêm các thuộc tính thời gian bổ sung cho Star Schema
    dim_date_df = (date_df
        .withColumn("dateKey", year(col("full_date")) * 10000 + month(col("full_date")) * 100 + dayofmonth(col("full_date")))
        .withColumn("year", year(col("full_date")))
        .withColumn("quarter", quarter(col("full_date")))
        .withColumn("month", month(col("full_date")))
        .withColumn("week", weekofyear(col("full_date")))
        .withColumn("day", dayofmonth(col("full_date")))
        .withColumn("day_of_year", dayofyear(col("full_date")))
        .withColumn("day_name_of_week", date_format(col("full_date"), "EEEE"))
        .withColumn("month_name_of_week", date_format(col("full_date"), "MMMM"))
        .select(
            "dateKey", "full_date", "year", "quarter", "month", 
            "week", "day", "day_of_year", "day_name_of_week", "month_name_of_week"
        )
    )

    # Ghi bảng Dim Date vào tầng Gold
    dim_date_df.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")
else:
    raise ValueError("Could not determine start and end dates from silver.clean_order")

# COMMAND ----------
# MAGIC %md ## 2. Fact Table

# COMMAND ----------
# DBTITLE 1, fact_table
# Đọc ngược lại các bảng Dim vừa tạo từ metastore (hoặc dùng thẳng DF phía trên)
dim_customer = spark.table("gold.dim_customer")
dim_seller = spark.table("gold.dim_seller")
dim_product = spark.table("gold.dim_product")
dim_order = spark.table("gold.dim_order")
dim_date = spark.table("gold.dim_date")

# Cập nhật tương tự cho các bảng silver tham chiếu ở bảng Fact
# COMMAND ----------
# DBTITLE 1, fact_table - Sửa lại

silver_cleaned_order_item = spark.table("silver.clean_order_item")
silver_cleaned_order = spark.table("silver.clean_order")
silver_cleaned_payment = spark.table("silver.clean_payment")
silver_cleaned_order_review = spark.table("silver.clean_order_review")

dim_customer = spark.table("gold.dim_customer")
dim_seller = spark.table("gold.dim_seller")
dim_product = spark.table("gold.dim_product")
dim_order = spark.table("gold.dim_order")
dim_date = spark.table("gold.dim_date")

# Base join order + order_item
base_df = silver_cleaned_order.join(
    silver_cleaned_order_item, 
    "order_id", 
    "inner"
)

# Join các dimension
fact_table_df = (base_df
    .join(dim_order, on="order_id", how="inner")
    .join(dim_product, on="product_id", how="inner")
    .join(dim_customer, on="customer_id", how="inner")
    .join(dim_seller, on="seller_id", how="inner")
    
    # Join payment (có thể nhiều payment → nên aggregate hoặc lấy latest)
    .join(silver_cleaned_payment, on="order_id", how="left")
    
    # Join review (thường 1 order có nhiều review → cần xử lý)
    .join(silver_cleaned_order_review, on="order_id", how="left")
    
    # === SỬA JOIN DIM_DATE Ở ĐÂY ===
    .join(
        dim_date,
        date_format(col("order_purchase_timestamp"), "yyyy-MM-dd") == date_format(col("full_date"), "yyyy-MM-dd"),
        how="inner"
    )
    .select(
        "order_id",
        "order_item_id",
        "customer_id",
        "product_id",
        "review_id",
        "seller_id",
        "dateKey",
        "price",
        "freight_value",
        "payment_value",
        "payment_installments",
        "payment_sequential"
    )
)

# Ghi bảng
fact_table_df.write.format("delta").mode("overwrite").saveAsTable("gold.fact_table")

print("Số dòng trong fact_table:", fact_table_df.count())

Số dòng trong fact_table: 112650


In [0]:
%sql 
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
quocluu,default


In [0]:
%sql
-- SHOW SCHEMAS IN data_dev_olist;
SHOW TABLES IN data_dev_olist.silver;

SHOW TABLES IN data_dev_olist.silver;

database,tableName,isTemporary
silver,clean_customer,false
silver,clean_geolocation,false
silver,clean_order,false
silver,clean_order_item,false
silver,clean_order_review,false
silver,clean_payment,false
silver,clean_product,false
silver,clean_product_category,false
silver,clean_seller,false
silver,date_dimension,false
